In [ ]:

# ============================================
# Task 3: Model Explainability with SHAP
# Fraud Detection Project
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import joblib

# --------------------------------------------
# 1. Load Model
# --------------------------------------------
# Update path if needed
MODEL_PATH = "../models/best_fraud_model.pkl"
model = joblib.load(MODEL_PATH)

print(" Model loaded successfully")

# --------------------------------------------
# 2. Load Prepared Test Data
# --------------------------------------------
# Assumes you saved processed data after Task 2
X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print(" Test data loaded")
print("X_test shape:", X_test.shape)

# --------------------------------------------
# 3. Built-in Feature Importance
# --------------------------------------------
feature_importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

plt.figure(figsize=(8, 5))
plt.barh(
    feature_importance_df["feature"][:10][::-1],
    feature_importance_df["importance"][:10][::-1]
)
plt.title("Top 10 Feature Importances (Model)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

# --------------------------------------------
# 4. SHAP Explainer Initialization
# --------------------------------------------
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# For binary classification (Fraud = 1)
shap_values = shap_values[1]

print(" SHAP values computed")

# --------------------------------------------
# 5. SHAP Global Summary Plot
# --------------------------------------------
shap.summary_plot(
    shap_values,
    X_test,
    plot_type="dot",
    show=True
)

# --------------------------------------------
# 6. SHAP Bar Plot (Mean Absolute Impact)
# --------------------------------------------
shap.summary_plot(
    shap_values,
    X_test,
    plot_type="bar",
    show=True
)

# --------------------------------------------
# 7. Identify Prediction Cases
# --------------------------------------------
y_pred = model.predict(X_test)

tp_index = np.where((y_test == 1) & (y_pred == 1))[0][0]
fp_index = np.where((y_test == 0) & (y_pred == 1))[0][0]
fn_index = np.where((y_test == 1) & (y_pred == 0))[0][0]

print(" Selected indices:")
print("True Positive index:", tp_index)
print("False Positive index:", fp_index)
print("False Negative index:", fn_index)

# --------------------------------------------
# 8. SHAP Force Plot – True Positive
# --------------------------------------------
shap.force_plot(
    explainer.expected_value[1],
    shap_values[tp_index],
    X_test.iloc[tp_index],
    matplotlib=True
)

# --------------------------------------------
# 9. SHAP Force Plot – False Positive
# --------------------------------------------
shap.force_plot(
    explainer.expected_value[1],
    shap_values[fp_index],
    X_test.iloc[fp_index],
    matplotlib=True
)

# --------------------------------------------
# 10. SHAP Force Plot – False Negative
# --------------------------------------------
shap.force_plot(
    explainer.expected_value[1],
    shap_values[fn_index],
    X_test.iloc[fn_index],
    matplotlib=True
)

# --------------------------------------------
# 11. SHAP Dependence Plot (Optional)
# --------------------------------------------
# Change feature name if needed
FEATURE_NAME = "time_since_signup"

if FEATURE_NAME in X_test.columns:
    shap.dependence_plot(
        FEATURE_NAME,
        shap_values,
        X_test
    )
else:
    print(f"⚠ Feature '{FEATURE_NAME}' not found in dataset")

print("✅ Task 3 SHAP Explainability completed")


: 